# 2. Semantic Clustering

BGE-M3 메시지 임베딩을 정규화한 뒤 UMAP/HDBSCAN/KMeans 기반 의미 군집을 탐색하고 최종 군집 라벨을 저장합니다.

## 입력 파일

- `data/processed/message_master_minimal.csv`
- `data/embeddings/message_embeddings_bge_m3_minimal.npy`

## 출력 파일

- `data/processed/message_master_minimal_final_clustered.csv`


## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.preprocessing import normalize
from sklearn.metrics import silhouette_score
from sklearn.cluster import KMeans

import umap
import hdbscan

## 2. Path configuration

In [ ]:
# =========================
# Project path configuration
# =========================

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
EMBEDDING_DIR = PROJECT_ROOT / "data" / "embeddings"
RESULTS_DIR = PROJECT_ROOT / "results" / "clustering"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

MESSAGE_MASTER_PATH = PROCESSED_DATA_DIR / "message_master_minimal.csv"
EMBEDDING_PATH = EMBEDDING_DIR / "message_embeddings_bge_m3_minimal.npy"
CLUSTERED_MESSAGE_PATH = PROCESSED_DATA_DIR / "message_master_minimal_final_clustered.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)

## 3. Load embedding data

In [ ]:
# =========================
# Load stage 1 outputs
# =========================

message_df = pd.read_csv(MESSAGE_MASTER_PATH)
embeddings = np.load(EMBEDDING_PATH)

print("message_df shape:", message_df.shape)
print("embeddings shape:", embeddings.shape)

assert len(message_df) == embeddings.shape[0], "message_df와 embeddings의 행 개수가 다릅니다."

## 4. Embedding normalization

In [ ]:
X = normalize(embeddings)

print("정규화 후 shape:", X.shape)
print("첫 번째 벡터 norm:", np.linalg.norm(X[0]))

## 5. UMAP dimensionality reduction

In [ ]:
umap_10d_model = umap.UMAP(
    n_neighbors=30,
    n_components=10,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

embedding_umap_10d = umap_10d_model.fit_transform(X)

print("UMAP 10D shape:", embedding_umap_10d.shape)

In [ ]:
umap_2d_model = umap.UMAP(
    n_neighbors=30,
    n_components=2,
    min_dist=0.1,
    metric="cosine",
    random_state=42
)

embedding_2d = umap_2d_model.fit_transform(X)

message_df["umap_x"] = embedding_2d[:, 0]
message_df["umap_y"] = embedding_2d[:, 1]

message_df[["umap_x", "umap_y"]].head()

## 6. HDBSCAN parameter check

In [ ]:
def test_hdbscan_params(
    cluster_input,
    min_cluster_size_list=[10, 15, 20, 30, 40, 50],
    min_samples_list=[5, 10, 15, 20]
):
    results = []

    for min_cluster_size in min_cluster_size_list:
        for min_samples in min_samples_list:
            
            model = hdbscan.HDBSCAN(
                min_cluster_size=min_cluster_size,
                min_samples=min_samples,
                metric="euclidean",
                cluster_selection_method="eom"
            )
            
            labels = model.fit_predict(cluster_input)
            probs = model.probabilities_
            
            n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
            outlier_count = np.sum(labels == -1)
            outlier_ratio = outlier_count / len(labels)
            
            valid_mask = labels != -1
            
            if n_clusters >= 2 and valid_mask.sum() > 1:
                sil_score = silhouette_score(
                    cluster_input[valid_mask],
                    labels[valid_mask],
                    metric="euclidean"
                )
                avg_prob = probs[valid_mask].mean()
            else:
                sil_score = np.nan
                avg_prob = np.nan
            
            results.append({
                "min_cluster_size": min_cluster_size,
                "min_samples": min_samples,
                "n_clusters": n_clusters,
                "outlier_count": outlier_count,
                "outlier_ratio": outlier_ratio,
                "avg_prob": avg_prob,
                "silhouette": sil_score
            })
    
    result_df = pd.DataFrame(results)
    
    return result_df

In [ ]:
hdbscan_result_df = test_hdbscan_params(
    cluster_input=embedding_umap_10d,
    min_cluster_size_list=[10, 15, 20, 30, 40, 50],
    min_samples_list=[5, 10, 15, 20]
)

display(hdbscan_result_df)

In [ ]:
display(
    hdbscan_result_df[
        hdbscan_result_df["n_clusters"].between(8, 15)
    ].sort_values(
        by=["n_clusters", "outlier_ratio", "avg_prob"],
        ascending=[True, True, False]
    )
)

In [ ]:
def apply_hdbscan(
    df,
    cluster_input,
    min_cluster_size,
    min_samples,
    cluster_col="hdbscan_cluster",
    prob_col="hdbscan_prob",
    outlier_col="hdbscan_outlier"
):
    model = hdbscan.HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=min_samples,
        metric="euclidean",
        cluster_selection_method="eom"
    )
    
    labels = model.fit_predict(cluster_input)
    probs = model.probabilities_
    
    df[cluster_col] = labels
    df[prob_col] = probs
    df[outlier_col] = df[cluster_col] == -1
    
    n_clusters = df[cluster_col].nunique() - (
        1 if -1 in df[cluster_col].unique() else 0
    )
    
    outlier_ratio = (df[cluster_col] == -1).mean()
    avg_prob = df.loc[df[cluster_col] != -1, prob_col].mean()
    
    print(f"HDBSCAN 결과: min_cluster_size={min_cluster_size}, min_samples={min_samples}")
    print(df[cluster_col].value_counts().sort_index())
    print()
    print("클러스터 수:", n_clusters)
    print("이상치 비율:", round(outlier_ratio, 4))
    print("평균 소속 확률:", round(avg_prob, 4))
    
    return df, model

## 7. KMeans clustering and evaluation

In [ ]:
message_df, hdbscan_model_10 = apply_hdbscan(
    df=message_df,
    cluster_input=embedding_umap_10d,
    min_cluster_size=10,
    min_samples=10,
    cluster_col="hdbscan_cluster_10",
    prob_col="hdbscan_prob_10",
    outlier_col="hdbscan_outlier_10"
)

In [ ]:
message_df, hdbscan_model_15 = apply_hdbscan(
    df=message_df,
    cluster_input=embedding_umap_10d,
    min_cluster_size=10,
    min_samples=5,
    cluster_col="hdbscan_cluster_15",
    prob_col="hdbscan_prob_15",
    outlier_col="hdbscan_outlier_15"
)

In [ ]:
def plot_cluster_result(
    df,
    cluster_col,
    title,
    noise_label=None
):
    plot_df = df.dropna(subset=[cluster_col, "umap_x", "umap_y"]).copy()
    plot_df[cluster_col] = plot_df[cluster_col].astype(int)
    
    clusters = sorted(plot_df[cluster_col].unique())
    
    if noise_label is not None:
        normal_clusters = [c for c in clusters if c != noise_label]
    else:
        normal_clusters = clusters
    
    cmap = plt.get_cmap("tab20", max(len(normal_clusters), 1))
    
    plt.figure(figsize=(12, 8))
    
    if noise_label is not None and noise_label in clusters:
        noise_df = plot_df[plot_df[cluster_col] == noise_label]
        plt.scatter(
            noise_df["umap_x"],
            noise_df["umap_y"],
            color="lightgray",
            s=16,
            alpha=0.45,
            label=f"Noise ({noise_label}), n={len(noise_df)}"
        )
    
    for i, cluster in enumerate(normal_clusters):
        cluster_df = plot_df[plot_df[cluster_col] == cluster]
        
        plt.scatter(
            cluster_df["umap_x"],
            cluster_df["umap_y"],
            color=cmap(i),
            s=18,
            alpha=0.75,
            label=f"Cluster {cluster}, n={len(cluster_df)}"
        )
        
        center_x = cluster_df["umap_x"].mean()
        center_y = cluster_df["umap_y"].mean()
        
        plt.text(
            center_x,
            center_y,
            str(cluster),
            fontsize=10,
            fontweight="bold",
            ha="center",
            va="center",
            bbox=dict(
                boxstyle="round,pad=0.25",
                facecolor="white",
                edgecolor="black",
                alpha=0.7
            )
        )
    
    plt.title(title, fontsize=15)
    plt.xlabel("UMAP-1")
    plt.ylabel("UMAP-2")
    plt.legend(
        title=cluster_col,
        bbox_to_anchor=(1.05, 1),
        loc="upper left",
        fontsize=9,
        markerscale=1.5
    )
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_cluster_result(
    df=message_df,
    cluster_col="hdbscan_cluster_10",
    title="HDBSCAN Result: min_cluster_size=10, min_samples=10",
    noise_label=-1
)

In [ ]:
plot_cluster_result(
    df=message_df,
    cluster_col="hdbscan_cluster_15",
    title="HDBSCAN Result: min_cluster_size=10, min_samples=5",
    noise_label=-1
)

In [ ]:
def test_kmeans_k(
    cluster_input,
    k_range=range(2, 21),
    silhouette_metric="cosine"
):
    results = []
    
    for k in k_range:
        model = KMeans(
            n_clusters=k,
            random_state=42,
            n_init=10
        )
        
        labels = model.fit_predict(cluster_input)
        
        inertia = model.inertia_
        sil_score = silhouette_score(
            cluster_input,
            labels,
            metric=silhouette_metric
        )
        
        results.append({
            "k": k,
            "inertia": inertia,
            "silhouette": sil_score
        })
        
        print(f"k={k}, inertia={inertia:.4f}, silhouette={sil_score:.4f}")
    
    return pd.DataFrame(results)

In [ ]:
kmeans_result_df = test_kmeans_k(
    cluster_input=X,
    k_range=range(2, 21),
    silhouette_metric="cosine"
)

display(kmeans_result_df)

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    kmeans_result_df["k"],
    kmeans_result_df["inertia"],
    marker="o"
)

plt.title("KMeans Elbow Method")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Inertia")
plt.xticks(kmeans_result_df["k"])
plt.grid(alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    kmeans_result_df["k"],
    kmeans_result_df["silhouette"],
    marker="o"
)

plt.title("KMeans Silhouette Score")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Silhouette Score")
plt.xticks(kmeans_result_df["k"])
plt.grid(alpha=0.3)
plt.show()

In [ ]:
selected_k = 3

kmeans_model = KMeans(
    n_clusters=selected_k,
    random_state=42,
    n_init=10
)

message_df["kmeans_cluster"] = kmeans_model.fit_predict(X)

print("KMeans 군집 결과")
print(message_df["kmeans_cluster"].value_counts().sort_index())

In [ ]:
for k in [10,11,12]:
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    
    message_df[f"kmeans_cluster_{k}"] = model.fit_predict(X)
    
    print(f"\nKMeans k={k}")
    print(message_df[f"kmeans_cluster_{k}"].value_counts().sort_index())

In [ ]:
plot_cluster_result(
    df=message_df,
    cluster_col="kmeans_cluster_10",
    title="KMeans Result: k=10",
    noise_label=None
)

plot_cluster_result(
    df=message_df,
    cluster_col="kmeans_cluster_11",
    title="KMeans Result: k=11",
    noise_label=None
)

plot_cluster_result(
    df=message_df,
    cluster_col="kmeans_cluster_12",
    title="KMeans Result: k=12",
    noise_label=None
)

## 8. Save final clustered message table

In [ ]:
message_df["final_cluster"] = message_df["kmeans_cluster_10"]

print("최종 군집 분포")
print(message_df["final_cluster"].value_counts().sort_index())

save_path = CLUSTERED_MESSAGE_PATH

message_df.to_csv(save_path, index=False, encoding="utf-8-sig")

print("저장 완료:", save_path)
print("저장된 데이터 크기:", message_df.shape)